<a href="https://colab.research.google.com/github/JeaLPaHu/telecomx-churn-analysis/blob/main/TelecomX_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📌 Extracción

Actividades a realizar en esta primera etapa:

1. Importar los datos
2. Comprender qué información contiene el dataset
3. Verificar las inconsistencias
4. Corregir las inconsistencias
5. Traducir las columnas y/o datos
6. Crear la columna de cuentas diarias

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [ ]:
url = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/refs/heads/main/TelecomX_Data.json'
response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)
df.head()

In [ ]:
df = pd.json_normalize(data)
df.head()

In [ ]:
print("=== INFORMACIÓN GENERAL ===")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.info()

## 🔧 Transformación

### VERIFICACIONES


In [ ]:
for col in df.columns:
    print(f'{col}: {df[col].nunique()}')
    if df[col].nunique() < 50:
        print(df[col].unique())
        print('-' * 50)

In [ ]:
print("Número de duplicados:", df.duplicated().sum())

In [ ]:
print("Número de Nulos:\n", df.isnull().sum())

In [ ]:
df.apply(lambda x: x.astype(str).str.strip() == '').sum()

In [ ]:
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')

In [ ]:
df = df[df['Churn'].str.strip() != '']
print("Filas después de eliminar Churn vacíos:", len(df))

In [ ]:
df.apply(lambda x: x.astype(str).str.strip() == '').sum()

In [ ]:
print("Número de Nulos:\n", df.isnull().sum())

In [ ]:
df = df.dropna(subset=['account.Charges.Total'])
print("Filas después de dropna:", len(df))

In [ ]:
df.to_csv('TelecomX_Data.csv', index=False)
print("✅ CSV exportado correctamente")

## OPCIONAL
### Estandarización al español

In [ ]:
columnas = {
    'customerID':                    'id',
    'Churn':                         'cancelacion',
    'customer.gender':               'genero',
    'customer.SeniorCitizen':        'tiene_mas_60',
    'customer.Partner':              'posee_pareja',
    'customer.Dependents':           'posee_dependientes',
    'customer.tenure':               'tiempo_contrato',
    'phone.PhoneService':            'servicio_telefonico',
    'phone.MultipleLines':           'multiples_lineas',
    'internet.InternetService':      'tipo_internet',
    'internet.OnlineSecurity':       'seguridad_online',
    'internet.OnlineBackup':         'backup_online',
    'internet.DeviceProtection':     'proteccion_dispositivo',
    'internet.TechSupport':          'soporte_tecnico',
    'internet.StreamingTV':          'streaming_tv',
    'internet.StreamingMovies':      'streaming_peliculas',
    'account.Contract':              'tipo_contrato',
    'account.PaperlessBilling':      'factura_digital',
    'account.PaymentMethod':         'metodo_pago',
    'account.Charges.Monthly':       'cargo_mensual',
    'account.Charges.Total':         'cargo_total',
}

df = df.rename(columns=columnas)
print("✅ Columnas renombradas:")
print(df.columns.tolist())

In [ ]:
df['cancelacion'] = df['cancelacion'].replace({'No': 'No', 'Yes': 'Sí'})
df['genero'] = df['genero'].replace({'Female': 'Femenino', 'Male': 'Masculino'})
df['posee_pareja'] = df['posee_pareja'].replace({'Yes': 'Sí', 'No': 'No'})
df['posee_dependientes'] = df['posee_dependientes'].replace({'Yes': 'Sí', 'No': 'No'})
df['factura_digital'] = df['factura_digital'].replace({'Yes': 'Sí', 'No': 'No'})
df['servicio_telefonico'] = df['servicio_telefonico'].replace({'Yes': 'Sí', 'No': 'No'})

In [ ]:
columnas_servicios = [
    'seguridad_online', 'backup_online', 'proteccion_dispositivo',
    'soporte_tecnico', 'streaming_tv', 'streaming_peliculas', 'multiples_lineas'
]
mapeo = {
    'No': 'No',
    'Yes': 'Sí',
    'No internet service': 'Sin servicio de internet',
    'No phone service': 'Sin servicio telefónico'
}
for col in columnas_servicios:
    df[col] = df[col].replace(mapeo)

In [ ]:
df['metodo_pago'] = df['metodo_pago'].replace({
    'Mailed check':            'Cheque por correo',
    'Electronic check':        'Cheque electrónico',
    'Credit card (automatic)': 'Tarjeta de crédito (automático)',
    'Bank transfer (automatic)':'Transferencia bancaria (automática)'
})
df.head()

In [ ]:
df['cuentas_diarias'] = (df['cargo_mensual'] / 30).round(2)
print("✅ Columna 'cuentas_diarias' creada")
df[['id', 'cargo_mensual', 'cuentas_diarias']].head()

In [ ]:
print(f"Dataset listo: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

## 📊 Carga y Análisis
### 1. Análisis Descriptivo

In [ ]:
df[['tiempo_contrato', 'cargo_mensual', 'cargo_total', 'cuentas_diarias']].describe().round(2)

In [ ]:
print("=== MEDIA POR GRUPO DE CANCELACIÓN ===")
df.groupby('cancelacion')[['tiempo_contrato', 'cargo_mensual', 'cargo_total', 'cuentas_diarias']].mean().round(2)

In [ ]:
print("=== MEDIANA ===")
print(df[['tiempo_contrato', 'cargo_mensual', 'cargo_total']].median().round(2))

print("\n=== DESVIACIÓN ESTÁNDAR ===")
print(df[['tiempo_contrato', 'cargo_mensual', 'cargo_total']].std().round(2))

In [ ]:
cat_cols = ['genero', 'tiene_mas_60', 'posee_pareja', 'posee_dependientes',
            'tipo_contrato', 'metodo_pago', 'tipo_internet']

for col in cat_cols:
    print(f"\n🔹 {col}:")
    print(df[col].value_counts())

###2. Distribución de Evasión

In [ ]:
print("=== DISTRIBUCIÓN DE CANCELACIÓN ===")
print(df['cancelacion'].value_counts())
print()
print(df['cancelacion'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
fig = px.histogram(
    df, x='cancelacion',
    color='cancelacion',
    text_auto=True,
    title='Distribución de Cancelación de Clientes',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'cancelacion': 'Cancelación', 'count': 'Número de Clientes'}
)
fig.update_layout(showlegend=False)
fig.show()

###3. Recuento por Variables Categóricas

In [ ]:
fig = px.histogram(
    df, x='genero', color='cancelacion',
    barmode='group', text_auto=True,
    title='Cancelación por Género',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'genero': 'Género', 'count': 'Clientes', 'cancelacion': 'Cancelación'}
)
fig.show()

In [ ]:
fig = px.histogram(
    df, x='tipo_contrato', color='cancelacion',
    barmode='group', text_auto=True,
    title='Cancelación por Tipo de Contrato',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'tipo_contrato': 'Tipo de Contrato', 'count': 'Clientes', 'cancelacion': 'Cancelación'}
)
fig.show()

In [ ]:
fig = px.histogram(
    df, x='tipo_internet', color='cancelacion',
    barmode='group', text_auto=True,
    title='Cancelación por Tipo de Internet',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'tipo_internet': 'Tipo de Internet', 'count': 'Clientes', 'cancelacion': 'Cancelación'}
)
fig.show()

In [ ]:
fig = px.histogram(
    df, x='metodo_pago', color='cancelacion',
    barmode='group', text_auto=True,
    title='Cancelación por Método de Pago',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'metodo_pago': 'Método de Pago', 'count': 'Clientes', 'cancelacion': 'Cancelación'}
)
fig.show()

In [ ]:
fig = px.histogram(
    df, x='tiempo_contrato', color='cancelacion',
    barmode='group', text_auto=True,
    title='Cancelación por Antigüedad',
    color_discrete_map={'No': '#2ecc71', 'Sí': '#e74c3c'},
    labels={'tiempo_contrato': 'Meses de Contrato', 'count': 'Clientes', 'cancelacion': 'Cancelación'}
)
fig.show()

## 📄 Informe Final

## 🔹 1. Introducción

TelecomX enfrenta una tasa de evasión (cancelación) del 26.5% sobre una base
de 7,043 clientes analizados. Este análisis tiene como objetivo identificar
los factores que llevan a la pérdida de clientes mediante un proceso ETL
completo y un Análisis Exploratorio de Datos (EDA).

## 🔹 2. Limpieza y Tratamiento de Datos

| Acción | Detalle |
|---|---|
| Carga de datos | JSON consumido desde API de GitHub con `requests` |
| Aplanamiento | Estructura anidada convertida con `pd.json_normalize()` |
| Churn vacío | Registros sin clasificación eliminados con `str.strip()` |
| TotalCharges | Convertida de string a float; NaN eliminados con `dropna()` |
| Traducción | Columnas y valores traducidos al español |
| Nueva variable | `cuentas_diarias` = cargo_mensual / 30 |

In [ ]:
print("=== RESUMEN EJECUTIVO ===\n")

total     = len(df)
churn_n   = df['cancelacion'].eq('Sí').sum()
churn_pct = churn_n / total * 100

print(f"  Total clientes analizados  : {total:,}")
print(f"  Clientes que cancelaron    : {churn_n:,} ({churn_pct:.1f}%)")
print(f"  Clientes que permanecieron : {total - churn_n:,} ({100 - churn_pct:.1f}%)")

print("\n--- Cargo mensual promedio ---")
print(df.groupby('cancelacion')['cargo_mensual'].mean().round(2))

print("\n--- Antigüedad promedio (meses) ---")
print(df.groupby('cancelacion')['tiempo_contrato'].mean().round(1))

print("\n--- Cancelación por tipo de contrato ---")
print(df.groupby('tipo_contrato')['cancelacion'].value_counts(normalize=True)
      .mul(100).round(1).unstack())

print("\n--- Cancelación por tipo de internet ---")
print(df.groupby('tipo_internet')['cancelacion'].value_counts(normalize=True)
      .mul(100).round(1).unstack())

print("\n--- Cancelación por método de pago ---")
print(df.groupby('metodo_pago')['cancelacion'].value_counts(normalize=True)
      .mul(100).round(1).unstack())

## 🔹 3. Conclusiones e Insights

### 🔴 Factores de alto riesgo identificados

**1. Tipo de contrato — el factor más crítico**
Los clientes con contrato mes a mes presentan una tasa de cancelación del ~42.7%,
frente al ~11% anual y ~2.8% bianual.

**2. Clientes nuevos (0-12 meses) en zona de peligro**
Cerca del 47.7% de los clientes con menos de 12 meses cancelan el servicio.
La etapa de onboarding es crítica para la retención.

**3. Cargos mensuales elevados**
Los clientes que se van pagan en promedio $74.44/mes versus $61.27/mes
de quienes permanecen.

**4. Fibra óptica con alta evasión**
A pesar de ser el servicio premium, los clientes de Fibra Óptica muestran
mayor cancelación que los de DSL.

**5. Cheque electrónico como señal de alerta**
Es el método de pago con mayor tasa de cancelación, asociado a clientes
menos comprometidos con la empresa.

**6. Adultos mayores más vulnerables**
Aunque son minoría, su tasa de cancelación es superior al promedio.


## 🔹 4. Recomendaciones Estratégicas

| # | Acción | Impacto |
|---|---|---|
| 1 | Incentivar contratos anuales/bianuales con descuentos exclusivos | 🔴 Alto |
| 2 | Programa de onboarding intensivo en los primeros 12 meses | 🔴 Alto |
| 3 | Revisar calidad del servicio de Fibra Óptica | 🟠 Medio |
| 4 | Bundling de TechSupport y OnlineSecurity en planes base | 🟠 Medio |
| 5 | Plan diferenciado para adultos mayores | 🟡 Medio |
| 6 | Migrar clientes de cheque electrónico a débito automático | 🟡 Medio |

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                   ANÁLISIS COMPLETADO ✅                        ║
║                                                                  ║
║  Dataset    : 7,043 clientes   Cancelación : 26.5%              ║
║  Variables  : 22 columnas      Gráficos    : 5 visualizaciones  ║
║                                                                  ║
║  Próximo paso → Modelo predictivo de Churn (ML)  🤖             ║
╚══════════════════════════════════════════════════════════════════╝
""")